# scipy.stats cheat sheet

**What's in here**
- Distribution objects: `pdf / cdf / ppf / rvs / fit`
- Descriptive stats: `describe`, skew, kurtosis
- Normality tests and why they always reject on large n
- t-tests, Mann-Whitney, KS test
- Correlation with p-values (`pearsonr`, `spearmanr`) vs `pandas.corr()`
- `linregress` vs `np.polyfit` vs sklearn
- Confidence intervals by hand and by bootstrap
- Autocorrelation by hand, effective sample size intuition
- Multiple testing note

Only numpy / pandas / scipy (no statsmodels).

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120); pd.set_option("display.max_columns", 30)
np.set_printoptions(precision=4, suppress=True)

df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"]).sort_values("time").reset_index(drop=True)
df["hour"] = df["time"].dt.hour
df["dow"] = df["time"].dt.dayofweek
df["weekend"] = df["dow"] >= 5
df.head()

,time,consumption_mwh,temp_c,wind_ms,solar_wm2,price_eur_mwh,hour,dow,weekend
0,2022-01-01 00:00:00+00:00,26858.4,0.11,7.00,0.0,81.83,0,5,True
1,2022-01-01 01:00:00+00:00,26177.8,-0.18,6.61,0.0,88.21,1,5,True
2,2022-01-01 02:00:00+00:00,26229.4,-1.11,7.14,0.0,84.71,2,5,True
3,2022-01-01 03:00:00+00:00,25381.3,-0.75,7.05,0.0,70.92,3,5,True
4,2022-01-01 04:00:00+00:00,25223.0,-0.09,7.36,0.0,60.78,4,5,True


## Distribution objects

Every distribution in `scipy.stats` has the same API. `loc` / `scale` shift and stretch it. Frozen distributions (`stats.norm(0, 1)`) are handy when you reuse parameters.

In [2]:
z = stats.norm(loc=0, scale=1)          # frozen standard normal
print("pdf(0)       =", z.pdf(0))
print("cdf(1.96)    =", z.cdf(1.96))
print("ppf(0.975)   =", z.ppf(0.975))    # inverse cdf -> quantile
print("sf(1.96)     =", z.sf(1.96))      # survival function = 1 - cdf (more precise in the tail)
print("mean/var     =", z.mean(), z.var())
print("5 samples    =", z.rvs(5, random_state=0))

pdf(0)       = 0.3989422804014327
cdf(1.96)    = 0.9750021048517795
ppf(0.975)   = 1.959963984540054
sf(1.96)     = 0.024997895148220435
mean/var     = 0.0 1.0
5 samples    = [1.7641 0.4002 0.9787 2.2409 1.8676]


Student-t has fatter tails than the normal - compare tail probabilities. Financial returns are usually closer to t with low degrees of freedom than to normal.

In [3]:
for dfree in [2, 3, 5, 30]:
    t = stats.t(df=dfree)
    print(f"t(df={dfree:2d}): P(|X|>3) = {2*t.sf(3):.4f}   vs normal {2*stats.norm.sf(3):.4f}")

t(df= 2): P(|X|>3) = 0.0955   vs normal 0.0027
t(df= 3): P(|X|>3) = 0.0577   vs normal 0.0027
t(df= 5): P(|X|>3) = 0.0301   vs normal 0.0027
t(df=30): P(|X|>3) = 0.0054   vs normal 0.0027


`fit` does maximum likelihood. For skewed positive quantities like consumption, try `lognorm`. Fix `floc=0` when you know the lower bound, otherwise the optimiser wastes a parameter.

In [4]:
x = df["consumption_mwh"].values
mu, sd = stats.norm.fit(x)
shape, loc, scale = stats.lognorm.fit(x, floc=0)
print(f"normal fit : mu={mu:.0f} sd={sd:.0f}")
print(f"lognorm fit: shape={shape:.3f} scale={scale:.0f}  (implied median={scale:.0f})")
# Which fits better? Compare log-likelihood
print("loglik normal :", stats.norm.logpdf(x, mu, sd).sum().round(0))
print("loglik lognorm:", stats.lognorm.logpdf(x, shape, loc, scale).sum().round(0))

normal fit : mu=29315 sd=4205
lognorm fit: shape=0.149 scale=29001  (implied median=29001)
loglik normal : -171046.0
loglik lognorm: -171518.0


## Descriptive statistics

`stats.describe` returns everything in one object. Note `skew` and `kurtosis` (Fisher: normal = 0). Pandas `.kurt()` is also excess kurtosis, but pandas uses the bias-corrected estimator, scipy's default is biased - the numbers differ slightly.

In [5]:
ret = df["price_eur_mwh"].diff()          # hourly price change (pct_change is unsafe: prices cross zero)
d = stats.describe(ret.dropna())
print(d)
print()
print("scipy  skew/kurt:", stats.skew(ret.dropna()).round(3), stats.kurtosis(ret.dropna()).round(3))
print("pandas skew/kurt:", ret.skew().round(3), ret.kurt().round(3))

DescribeResult(nobs=17519, minmax=(-255.7, 253.99), mean=-0.0014275928991381046, variance=584.974289827372, skewness=-0.12997865610954717, kurtosis=22.730776409735036)

scipy  skew/kurt: -0.13 22.731
pandas skew/kurt: -0.13 22.738


**Pitfall:** `pct_change()` on a series that can be zero or negative (power prices!) gives infinities and sign flips. Use differences or check the sign first.

**Interview check:** "The kurtosis is 40 - what does that tell you?" Extremely fat tails: a few spike hours dominate the variance. Any least-squares fit will be driven by those hours.

In [6]:
pct = df["price_eur_mwh"].pct_change()
print("n inf in pct_change:", np.isinf(pct).sum(), " |  n negative prices:", (df["price_eur_mwh"] <= 0).sum())
print("largest |pct_change|:", pct.abs().replace(np.inf, np.nan).max().round(1))

n inf in pct_change: 0  |  n negative prices: 43
largest |pct_change|: 272.7


## Normality tests

`shapiro` is capped at n=5000 (warns above). `jarque_bera` works on any n. Both reject for almost any real dataset once n is large - a test tells you *whether* there is a deviation, not whether it *matters*. Look at QQ plots / quantiles instead.

In [7]:
sample = ret.dropna().sample(2000, random_state=0)
print("shapiro     :", stats.shapiro(sample))
print("jarque_bera :", stats.jarque_bera(ret.dropna()))
print("normaltest  :", stats.normaltest(ret.dropna()))

# Even a tiny deviation gets "significant" as n grows
rng = np.random.default_rng(0)
for n in [100, 1000, 10_000, 100_000]:
    x = rng.standard_t(df=30, size=n)         # almost normal
    print(f"n={n:>7d}  jarque_bera p = {stats.jarque_bera(x).pvalue:.4f}")

shapiro     : ShapiroResult(statistic=0.801366691368882, pvalue=3.4361554180226804e-44)
jarque_bera : SignificanceResult(statistic=377210.18352470326, pvalue=0.0)
normaltest  : NormaltestResult(statistic=4540.024382762284, pvalue=0.0)
n=    100  jarque_bera p = 0.6611
n=   1000  jarque_bera p = 0.4534
n=  10000  jarque_bera p = 0.0038
n= 100000  jarque_bera p = 0.0000


A quantile comparison is more informative than a p-value: compare empirical quantiles to the fitted normal's.

In [8]:
q = np.array([0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999])
r = ret.dropna()
emp = np.quantile(r, q)
theo = stats.norm.ppf(q, r.mean(), r.std())
pd.DataFrame({"q": q, "empirical": emp.round(1), "normal": theo.round(1), "ratio": (emp/theo).round(2)})

,q,empirical,normal,ratio
0,0.001,-198.0,-74.7,2.65
1,0.010,-46.1,-56.3,0.82
2,0.050,-30.8,-39.8,0.77
3,0.500,-0.2,-0.0,126.09
4,0.950,31.2,39.8,0.79
5,0.990,48.9,56.3,0.87
6,0.999,188.5,74.7,2.52


## Two-sample tests: is weekend consumption different?

`ttest_ind` assumes independent samples; `equal_var=False` (Welch) is the safer default. Hourly data is heavily autocorrelated so the p-value is far too optimistic - see the effective-sample-size section below.

In [9]:
wd = df.loc[~df["weekend"], "consumption_mwh"]
we = df.loc[df["weekend"], "consumption_mwh"]
print(f"means: weekday={wd.mean():.0f}  weekend={we.mean():.0f}  diff={wd.mean()-we.mean():.0f}")
print("Welch t-test :", stats.ttest_ind(wd, we, equal_var=False))
print("Mann-Whitney :", stats.mannwhitneyu(wd, we))          # rank-based, no normality assumption
print("KS 2-sample  :", stats.ks_2samp(wd, we))              # any difference in distribution shape

means: weekday=29914  weekend=27832  diff=2082
Welch t-test : TtestResult(statistic=30.651345029032704, pvalue=8.699101377521837e-197, df=9450.774114384256)
Mann-Whitney : MannwhitneyuResult(statistic=40218392.5, pvalue=4.2932929940585315e-184)
KS 2-sample  : KstestResult(statistic=0.2229090354090354, pvalue=3.5265171364378924e-157, statistic_location=28763.9, statistic_sign=-1)


One-sample test: is the mean hourly price change zero? `ttest_1samp`.

In [10]:
res = stats.ttest_1samp(ret.dropna(), popmean=0)
print(res)
print("95% CI for mean change:", res.confidence_interval(0.95))

TtestResult(statistic=-0.007812511074732353, pvalue=0.9937666704025488, df=17518)
95% CI for mean change: ConfidenceInterval(low=-0.35959975759230967, high=0.3567445717940334)


**Interview check:** "The t-test says weekday and weekend differ with p=1e-300. Do you believe it?" The direction and size, yes - the p-value, no. Hourly observations within a day are nearly identical, so the effective number of independent observations is far smaller than 17,520. Also the hour-of-day mix is the same in both groups here, but in general check that you are comparing like with like (confounders).

## Correlation with p-values

`pearsonr` / `spearmanr` return the statistic and p-value. `pandas.corr()` gives only the matrix. Spearman is rank-based - robust to outliers and monotone non-linearity.

In [11]:
sub = df[["consumption_mwh", "temp_c", "wind_ms", "price_eur_mwh"]].dropna()
print("pearson  cons~temp :", stats.pearsonr(sub["consumption_mwh"], sub["temp_c"]))
print("spearman cons~temp :", stats.spearmanr(sub["consumption_mwh"], sub["temp_c"]))
print("kendall  cons~temp :", stats.kendalltau(sub["consumption_mwh"], sub["temp_c"]))
print()
print(sub.corr().round(3))
print()
print(sub.corr(method="spearman").round(3))

pearson  cons~temp : PearsonRResult(statistic=-0.2208429261510589, pvalue=1.7071375032331722e-192)
spearman cons~temp : SignificanceResult(statistic=-0.21396760378039317, pvalue=1.5210996237140362e-180)
kendall  cons~temp : SignificanceResult(statistic=-0.14583698813644874, pvalue=3.126071837322528e-184)

                 consumption_mwh  temp_c  wind_ms  price_eur_mwh
consumption_mwh            1.000  -0.221    0.031          0.576
temp_c                    -0.221   1.000   -0.047         -0.140
wind_ms                    0.031  -0.047    1.000         -0.307
price_eur_mwh              0.576  -0.140   -0.307          1.000

                 consumption_mwh  temp_c  wind_ms  price_eur_mwh
consumption_mwh            1.000  -0.214    0.022          0.597
temp_c                    -0.214   1.000   -0.044         -0.153
wind_ms                    0.022  -0.044    1.000         -0.297
price_eur_mwh              0.597  -0.153   -0.297          1.000


**Pitfall:** a low Pearson correlation does not mean no relationship. Consumption vs temperature is V-shaped (heating below 15C, cooling above 22C), so the linear correlation understates the dependence. Bin and look.

In [12]:
bins = pd.cut(df["temp_c"], bins=[-10, 0, 5, 10, 15, 20, 25, 30])
df.groupby(bins, observed=True)["consumption_mwh"].agg(["mean", "count"]).round(0)

,mean,count
temp_c,,
"(-10, 0]",30060.0,1144
"(0, 5]",30962.0,3671
"(5, 10]",30363.0,3985
"(10, 15]",27431.0,4070
"(15, 20]",28219.0,3578
"(20, 25]",29784.0,1025
"(25, 30]",30142.0,47


## Simple regression three ways

`stats.linregress`, `np.polyfit`, and sklearn `LinearRegression` give identical slope/intercept. Only `linregress` gives standard errors and p-values out of the box.

In [13]:
from sklearn.linear_model import LinearRegression

x = sub["temp_c"].values; y = sub["consumption_mwh"].values
lr = stats.linregress(x, y)
print("linregress :", f"slope={lr.slope:.2f} intercept={lr.intercept:.1f} r={lr.rvalue:.3f} p={lr.pvalue:.2e} stderr={lr.stderr:.2f}")
b1, b0 = np.polyfit(x, y, deg=1)
print("polyfit    :", f"slope={b1:.2f} intercept={b0:.1f}")
m = LinearRegression().fit(x.reshape(-1, 1), y)
print("sklearn    :", f"slope={m.coef_[0]:.2f} intercept={m.intercept_:.1f}")

linregress : slope=-138.82 intercept=30693.9 r=-0.221 p=1.71e-192 stderr=4.63
polyfit    : slope=-138.82 intercept=30693.9


sklearn    : slope=-138.82 intercept=30693.9


## Confidence intervals

By hand: `mean +- t_crit * std/sqrt(n)`. `stats.t.interval` does the same. Both assume i.i.d. observations.

In [14]:
x = we.values
n = len(x); m_ = x.mean(); se = x.std(ddof=1) / np.sqrt(n)
t_crit = stats.t.ppf(0.975, df=n - 1)
print(f"by hand     : {m_ - t_crit*se:.1f} .. {m_ + t_crit*se:.1f}")
print("t.interval  :", np.round(stats.t.interval(0.95, df=n - 1, loc=m_, scale=se), 1))
print("norm approx :", np.round(stats.norm.interval(0.95, loc=m_, scale=se), 1))

by hand     : 27719.9 .. 27943.8
t.interval  : [27719.9 27943.8]
norm approx : [27720.  27943.7]


Bootstrap CI for any statistic (median, skew, a ratio...). `stats.bootstrap` needs the data as a tuple of samples and a statistic that accepts `axis`.

In [15]:
r = ret.dropna().values
bs = stats.bootstrap((r,), np.median, confidence_level=0.95, n_resamples=2000, random_state=0, method="percentile")
print("median          :", np.median(r).round(3))
print("bootstrap 95% CI:", bs.confidence_interval)

# manual bootstrap for a custom statistic: 99th percentile of price changes
rng = np.random.default_rng(0)
boot = np.array([np.quantile(rng.choice(r, len(r)), 0.99) for _ in range(1000)])
print(f"99th pct: {np.quantile(r, 0.99):.1f}   bootstrap CI: {np.quantile(boot, [0.025, 0.975]).round(1)}")

median          : -0.18
bootstrap 95% CI: ConfidenceInterval(low=-0.5400000000000063, high=0.17000000000001592)


99th pct: 48.9   bootstrap CI: [46.9 50.7]


**Pitfall:** the i.i.d. bootstrap destroys autocorrelation. For time series use a *block* bootstrap (resample contiguous blocks) - here is the idea in five lines.

In [16]:
def block_bootstrap_mean(x, block=24, n_boot=500, seed=0):
    rng = np.random.default_rng(seed)
    n = len(x); n_blocks = int(np.ceil(n / block))
    out = []
    for _ in range(n_boot):
        starts = rng.integers(0, n - block, n_blocks)
        sample = np.concatenate([x[s:s + block] for s in starts])[:n]
        out.append(sample.mean())
    return np.array(out)

x = df["consumption_mwh"].values
iid = np.array([np.random.default_rng(i).choice(x, len(x)).mean() for i in range(300)])
blk = block_bootstrap_mean(x, block=24*7, n_boot=300)
print(f"SE of the mean: iid bootstrap={iid.std():.1f}   weekly-block bootstrap={blk.std():.1f}")

SE of the mean: iid bootstrap=31.3   weekly-block bootstrap=187.6


## Autocorrelation by hand

No statsmodels: correlate the series with a shifted copy. `pandas.Series.autocorr(lag)` does the same in one call.

In [17]:
s = df["consumption_mwh"]
def acf(series, lags):
    return pd.Series({k: np.corrcoef(series[k:], series[:-k])[0, 1] for k in lags})

lags = [1, 2, 3, 6, 12, 24, 48, 168]
a = acf(s.values, lags)
print(a.round(3))
print("pandas autocorr(24):", s.autocorr(24).round(3))

1      0.935
2      0.781
3      0.584
6      0.084
12    -0.047
24     0.914
48     0.867
168    0.930
dtype: float64
pandas autocorr(24): 0.914


**Interview check:** "The correlation between consumption and temperature is -0.5 with p ~ 0. Is it significant given autocorrelation?"

With lag-1 autocorrelation rho in both series, the effective sample size is roughly n_eff = n (1 - rho_x rho_y) / (1 + rho_x rho_y). If rho ~ 0.98 in both, n_eff collapses to a few percent of n. Recompute the standard error of r with n_eff: still significant here, but the p-value moves from 1e-300 to something honest - and for weaker correlations the conclusion can flip.

In [18]:
x = sub["consumption_mwh"].values; y = sub["temp_c"].values
n = len(x)
rho_x = np.corrcoef(x[1:], x[:-1])[0, 1]; rho_y = np.corrcoef(y[1:], y[:-1])[0, 1]
n_eff = n * (1 - rho_x * rho_y) / (1 + rho_x * rho_y)
r = np.corrcoef(x, y)[0, 1]
def p_for_r(r, n):
    t = r * np.sqrt((n - 2) / (1 - r**2))
    return 2 * stats.t.sf(abs(t), df=n - 2)
print(f"rho_x={rho_x:.3f} rho_y={rho_y:.3f}  n={n}  n_eff={n_eff:.0f}")
print(f"r={r:.3f}   p(naive n)={p_for_r(r, n):.2e}   p(n_eff)={p_for_r(r, n_eff):.2e}")
# A weaker correlation where the conclusion flips:
r_weak = 0.03
print(f"r={r_weak}: p(naive)={p_for_r(r_weak, n):.3f}   p(n_eff)={p_for_r(r_weak, n_eff):.3f}")

rho_x=0.935 rho_y=0.992  n=17520  n_eff=654
r=-0.221   p(naive n)=1.71e-192   p(n_eff)=1.13e-08
r=0.03: p(naive)=0.000   p(n_eff)=0.444


Another honest check: correlate the *changes* (or residuals after removing hour-of-day/seasonality). Two trending series are correlated by construction.

Here the sign even flips: hour to hour, temperature and consumption both rise in the morning and fall at night (diurnal cycle, positive correlation), while across days colder weather means more heating demand (negative correlation). Two mechanisms at two frequencies - one number cannot summarise both.

In [19]:
dx = np.diff(x); dy = np.diff(y)
print("corr of levels :", np.corrcoef(x, y)[0, 1].round(3))
print("corr of changes:", np.corrcoef(dx, dy)[0, 1].round(3))

corr of levels : -0.221
corr of changes: 0.253


## Multiple testing

If you test 20 features at alpha=0.05 you expect one false positive. Bonferroni (`alpha/m`) is conservative; Benjamini-Hochberg (`stats.false_discovery_control`) controls the false discovery rate.

In [20]:
rng = np.random.default_rng(1)
y = df["consumption_mwh"].values
noise_feats = rng.normal(size=(len(y), 20))            # 20 features of pure noise
pvals = np.array([stats.pearsonr(noise_feats[:, j], y).pvalue for j in range(20)])
print("raw p < 0.05        :", (pvals < 0.05).sum(), "of 20 noise features")
print("bonferroni p < 0.05 :", (pvals < 0.05 / 20).sum())
print("BH adjusted p < 0.05:", (stats.false_discovery_control(pvals) < 0.05).sum())

raw p < 0.05        : 1 of 20 noise features
bonferroni p < 0.05 : 0
BH adjusted p < 0.05: 0


**Interview check:** "You tried 30 lag/rolling-window combinations and one has t-stat 2.5. Is it real?" Not on that evidence: with 30 tries, a max |t| of 2.5 is roughly what you expect from noise. Fix the specification on a training period, or correct for the number of trials, or confirm on held-out data.

## Quick reference

| Task | Call |
|---|---|
| Quantile of a normal | `stats.norm.ppf(q, loc, scale)` |
| Tail probability | `stats.norm.sf(x)` |
| Fit a distribution | `stats.lognorm.fit(x, floc=0)` |
| Normality | `stats.jarque_bera(x)`, `stats.shapiro(x[:5000])` |
| Two means | `stats.ttest_ind(a, b, equal_var=False)` |
| Two distributions | `stats.mannwhitneyu(a, b)`, `stats.ks_2samp(a, b)` |
| Correlation + p | `stats.pearsonr(x, y)`, `stats.spearmanr(x, y)` |
| OLS with SEs | `stats.linregress(x, y)` |
| Bootstrap CI | `stats.bootstrap((x,), np.median)` |
| Multiple tests | `stats.false_discovery_control(pvals)` |